# Notebook 06: MH-DDPM Fine-tuning on Real Spectra
## Fine-tune DDPM on 379 real contaminated spectra + MMD/JSD Validation

In [ ]:
import numpy as np, pandas as pd, struct
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch; torch.manual_seed(42)
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
OUT = BASE / 'data' / 'processed'
FIG = BASE / 'figures'; FIG.mkdir(exist_ok=True)
df = pd.read_parquet(OUT / 'real_dataset.parquet')

In [ ]:
def unpack(row):    n = row['n_wl']; data = struct.unpack(f'{n*2}d', row['spectrum_bytes'])    return np.array(data[::2]), np.array(data[1::2])target_wl = np.arange(230, 610, 1)def interp(row): return np.interp(target_wl, *unpack(row))X_contam = np.vstack(df[df['label']==1].apply(interp, axis=1).values)print(f'Contaminated spectra for DDPM training: {X_contam.shape}')

## Simple DDPM implementation

In [ ]:
class SimpleDDPM(nn.Module):
    def __init__(self, dim, timesteps=100):
        super().__init__()
        self.T = timesteps
        self.beta = torch.linspace(1e-4, 0.02, timesteps)
        self.alpha = 1 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, 0)
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        t_norm = t.float() / self.T
        return self.net(torch.cat([x, t_norm.unsqueeze(1)], dim=1))
    
    @torch.no_grad()
    def sample(self, n_samples, dim):
        x = torch.randn(n_samples, dim)
        for t in reversed(range(self.T)):
            z = torch.randn_like(x) if t > 0 else 0
            t_tensor = torch.full((n_samples,), t, dtype=torch.long, device=x.device)
            eps = self(x, t_tensor)
            x = (x - (1-self.alpha[t])/torch.sqrt(1-self.alpha_bar[t]) * eps) / torch.sqrt(self.alpha[t]) + torch.sqrt(self.beta[t]) * z
        return x

device = 'cpu'
ddpm = SimpleDDPM(X_contam.shape[1], timesteps=50).to(device)
opt = torch.optim.Adam(ddpm.parameters(), lr=1e-3)

X_t = torch.FloatTensor(X_contam).to(device)
dl = DataLoader(TensorDataset(X_t), batch_size=32, shuffle=True)

for epoch in range(30):
    total_loss = 0
    for (batch,) in dl:
        opt.zero_grad()
        t = torch.randint(0, ddpm.T, (batch.shape[0],), device=device)
        noise = torch.randn_like(batch)
        x_noisy = torch.sqrt(ddpm.alpha_bar[t]) * batch + torch.sqrt(1 - ddpm.alpha_bar[t]) * noise
        pred = ddpm(x_noisy, t)
        loss = nn.MSELoss()(pred, noise)
        loss.backward(); opt.step()
        total_loss += loss.item()
    if epoch % 5 == 0: print(f'  Epoch {epoch}: loss={total_loss/len(dl):.6f}')

## Generate synthetic spectra

In [ ]:
synthetic = ddpm.sample(50, X_contam.shape[1]).numpy()
print(f'Generated {len(synthetic)} synthetic spectra')

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(min(10, len(synthetic))):
    ax.plot(target_wl, synthetic[i], alpha=0.5, linewidth=1)
for i in range(min(10, len(X_contam))):
    ax.plot(target_wl, X_contam[i], alpha=0.3, linewidth=1, color='red')
ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Absorbance')
ax.set_title('Real (red) vs Synthetic (blue) Spectra', fontsize=13)
plt.tight_layout()
fig.savefig(FIG / 'ddpm_synthetic_vs_real.png', dpi=300)
print('Saved: ddpm_synthetic_vs_real.png')
plt.close()

## MMD and JSD Validation

In [ ]:
def mmd(x, y, kernel='rbf', gamma=1.0):
    Kxx = torch.exp(-gamma * torch.cdist(x, x)**2).mean()
    Kyy = torch.exp(-gamma * torch.cdist(y, y)**2).mean()
    Kxy = torch.exp(-gamma * torch.cdist(x, y)**2).mean()
    return Kxx + Kyy - 2*Kxy

def jsd(p, q, bins=50):
    hist_p, _ = np.histogram(p, bins=bins, density=True)
    hist_q, _ = np.histogram(q, bins=bins, density=True)
    p_norm = hist_p / (hist_p.sum() + 1e-10)
    q_norm = hist_q / (hist_q.sum() + 1e-10)
    m = 0.5 * (p_norm + q_norm)
    return 0.5 * (np.sum(p_norm * np.log(p_norm / m + 1e-10)) + np.sum(q_norm * np.log(q_norm / m + 1e-10)))

x_real = torch.FloatTensor(X_contam[:100])
x_synth = torch.FloatTensor(synthetic[:100])
mmd_val = mmd(x_real, x_synth, gamma=0.001)
print(f'MMD: {mmd_val:.6f}')

jsd_val = jsd(X_contam.mean(axis=1), synthetic.mean(axis=1))
print(f'JSD: {jsd_val:.6f}')
print(f'JSD < 0.1: {"✅ PASS" if jsd_val < 0.1 else "❌ FAIL"}')
print('\n✅ DDPM fine-tuning complete!')